# How much does choosing a configuration help?

In this notebook, we walk through measuring how much is gained by choosing among the benchmark's private SCE configurations instead of using one default, and whether a private choice could pay for its own budget. At each ε, we compare:

1. a default configuration fixed in advance;
2. one shared configuration, chosen on the whole benchmark;
3. a separate configuration for each instance, an oracle no private method could match;
4. the same oracle, choosing on the trials it is scored on.

Seeds 0–4 of the released results choose, and seeds 5–9 score.

## Opening the SCE collection

We fetch and open the release as in [Compare on the benchmark](compare_on_the_benchmark.ipynb).

In [ ]:
from pathlib import Path

from dpstats import fetch_benchmark, open_collection
from dpstats.tasks import SingleCoefficientEstimation

workspace = fetch_benchmark(Path("dp-benchmark"))
if open_collection(workspace, SingleCoefficientEstimation).load_sweep("current").get("algorithms") == []:
    raise ValueError("SCE Distribution results are pending in this revision.")

## Choosing the pool

The pool is every released configuration with the standard privacy guarantee, leaving out those tuned on this benchmark's data (`tuned@` in their name). The default is `AdaSSP, scaled`, fixed before reading any cost.

In [ ]:
from dpstats import open_collection
from dpstats.algorithm import PrivacyInterpretation
from dpstats.tasks import SingleCoefficientEstimation

collection = open_collection(workspace, SingleCoefficientEstimation)
sweep = collection.load_sweep("current")
pool = [
    algorithm
    for algorithm in collection.load_algorithms(source="release").values()
    if algorithm.privacy_interpretation is PrivacyInterpretation.STANDARD and "tuned@" not in algorithm.label
]
DEFAULT = "AdaSSP, scaled"
weights = collection.load_weights()
print(f"release {collection.release.version}: {len(weights)} instances, {len(pool)} configurations")
print("budgets:", [budget.eps for budget in sweep["budgets"]])
print("seeds:", sweep["seeds"])

## Measuring cost

Cost is `absolute_error_to_ols` divided by `published_se`, as in [Compare on the benchmark](compare_on_the_benchmark.ipynb); a trial succeeds when its cost is at most 1. Instances carry the release's weights.

In [ ]:
from dpstats.analysis import measure_results, compute_error_ratio
from dpstats.tasks import list_built_in_metrics

metric = list_built_in_metrics(SingleCoefficientEstimation)["absolute_error_to_ols"]
results = collection.load_results(sweep, algorithms=pool)
trials = compute_error_ratio(measure_results(results, metric), "published_se", metric=metric)
trials["eps"] = trials["budget"].map(lambda budget: budget.eps)
trials["success"] = trials[metric.name] <= 1
trials["half"] = trials["alg_seed"].map(lambda seed: "tuning" if seed < 5 else "evaluation")
print(trials["half"].value_counts().to_string())

## Making the shared choice

At each ε, the shared choice is the configuration with the highest weighted success rate on the tuning seeds.

In [ ]:
from dpstats.analysis import compute_profile_curves

scores = (
    trials[trials["half"] == "tuning"]
    .groupby("eps")
    .apply(lambda rows: compute_profile_curves(rows, [1.0], metric=metric, weights=weights))
)
shared = scores.loc[scores.groupby("eps")["fraction"].idxmax()].droplevel(1)
shared[["algorithm_label", "fraction"]].round(3)

## Making the per-instance choices

On each instance and ε, the oracle takes the configuration with the most successful trials, breaking ties by the lower median cost. On the tuning seeds this is the oracle; on the evaluation seeds it is the oracle choosing on its own trials.

In [ ]:
best = (
    trials.groupby(["half", "eps", "instance_uuid", "algorithm_label"])
    .agg(share=("success", "mean"), median=(metric.name, "median"))
    .reset_index()
    .sort_values(["share", "median"], ascending=[False, True])
    .groupby(["half", "eps", "instance_uuid"])
    .head(1)
)

## Scoring on the evaluation seeds

Each row is one ε and each column one way of choosing; the values are weighted success rates on seeds 5–9.

In [ ]:
import pandas as pd

evaluation = trials[trials["half"] == "evaluation"]
cells = evaluation[["eps", "instance_uuid"]].drop_duplicates()
KEYS = ["eps", "instance_uuid", "algorithm_label"]
choices = pd.concat(
    {
        "default": cells.assign(algorithm_label=DEFAULT),
        "shared choice": cells.merge(shared["algorithm_label"], on="eps"),
        "per-instance oracle": best.loc[best["half"] == "tuning", KEYS],
        "oracle on its own trials": best.loc[best["half"] == "evaluation", KEYS],
    },
    names=["choice"],
).reset_index("choice")
chosen = evaluation.merge(choices, on=KEYS)
chosen["algorithm"] = chosen["algorithm_label"] = chosen["choice"]
success = (
    chosen.groupby("eps")
    .apply(lambda rows: compute_profile_curves(rows, [1.0], metric=metric, weights=weights))
    .pivot_table(index="eps", columns="algorithm", values="fraction")[choices["choice"].unique()]
    .rename_axis(columns="choice")
)
success.round(3)

1. The shared choice beats the default at every ε.
2. The oracle is well above the shared choice.
3. Choosing on the scored trials inflates the oracle further, since it also picks lucky noise.

## Plotting the choices at ε = 1

In [ ]:
import numpy as np

from dpstats.analysis import plot_profiles

curves = compute_profile_curves(
    chosen[chosen["eps"] == 1.0], np.geomspace(0.1, 100.0, 31), metric=metric, weights=weights
)
figure = plot_profiles(
    curves,
    xlabel="absolute error / published_se",
    ylabel="share of trials within τ",
    xscale="log",
    tau=1.0,
)
figure.show()

## Bounding a private choice

[DPSelectionLiuTalwar](https://saminulh.github.io/dpstats/library/reference/dp_tuning/dpstats.dp_tuning.DPSelectionLiuTalwar/) is 3ε-DP when each candidate runs at ε, so at total budget ε no such selector beats the oracle run at ε / 3. We read the oracle one budget lower: exactly ε / 3 at total 3, slightly below it at totals 1 and 10.

In [ ]:
bound = success[["shared choice", "default"]].assign(
    **{"per-instance oracle one budget lower": success["per-instance oracle"].shift()}
)
bound.dropna().rename_axis("total ε").round(3)

Where the bound sits above the shared choice, a private per-instance selector is not ruled out, though a real one also pays for noise in its scores.

## Reading which configurations the oracle picks

The four most picked configurations at each ε, by weighted share of instances. `Zero` succeeds where the coefficient lies within one standard error of zero.

In [ ]:
(
    best[best["half"] == "tuning"]
    .merge(weights, on="instance_uuid")
    .groupby(["eps", "algorithm_label"])["weight"]
    .sum()
    .sort_values(ascending=False)
    .groupby("eps")
    .head(4)
    .sort_index(level="eps", sort_remaining=False)
    .rename("share of instances")
    .round(3)
    .to_frame()
)

## Limits of this comparison

1. A wider grid of settings could change both the oracle and the shared choice.
2. The oracle chooses from five seeds per instance, so its choice carries noise.
3. The bound ignores the selector's own noise.
4. Seeds split trials of the same data, not records, so no choice here is private.